# Step 3: Publish

Pushes best model to HuggingFace + GGUF export.

**Secrets needed:** `HF_TOKEN`
**Input:** `merged_*/` and `eval_results/` on Drive (from notebook 02)

In [ ]:
# Install — restart runtime after!
!pip install unsloth --upgrade --quiet
print('Done. Now: Runtime → Restart session, then run next cell.')

In [ ]:
# Setup (after restart)
from google.colab import drive, userdata
drive.mount('/content/drive')

import os, json, glob
DRIVE = '/content/drive/MyDrive/distillreasoning'

from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'))
print('Logged into HuggingFace ✅')

In [ ]:
# Load eval results and pick best model
comparison_file = f'{DRIVE}/eval_results/comparison.json'
assert os.path.exists(comparison_file), 'Run 02_eval.ipynb first!'

all_results = json.load(open(comparison_file))

# Filter to our distilled/GRPO models
our_models = {k: v for k, v in all_results.items() if 'sft' in k or 'grpo' in k}

print('Our models:')
for name, scores in our_models.items():
    gsm = scores.get('gsm8k_cot', '?')
    math = scores.get('minerva_math', '?')
    print(f'  {name:25s} GSM8K={gsm}% MATH={math}%')

best = max(our_models, key=lambda k: our_models[k].get('gsm8k_cot', 0))
print(f'\nBest model: {best}')
print(f'Scores: {our_models[best]}')

In [ ]:
# Push best model to HuggingFace
from unsloth import FastLanguageModel

best_dir = f'{DRIVE}/merged_{best}'
model, tok = FastLanguageModel.from_pretrained(
    model_name=best_dir, max_seq_length=4096, dtype=None, load_in_4bit=False,
)

HF_REPO = 'bmeyer2025/qwen3.5-4b-reasoning-distilled'

model.push_to_hub_merged(
    HF_REPO, tok,
    save_method='merged_16bit',
    token=userdata.get('HF_TOKEN'),
)
print(f'Pushed merged model to {HF_REPO} ✅')

In [ ]:
# GGUF export
model.push_to_hub_gguf(
    HF_REPO, tok,
    quantization_method=['q4_k_m', 'q8_0'],
    token=userdata.get('HF_TOKEN'),
)
print(f'GGUF pushed to {HF_REPO} ✅')
print(f'\nRun locally:')
print(f'  ollama run hf.co/{HF_REPO}:Q4_K_M')